# Initial Data Inspection

**Dataset:** UCI Student Performance — Portuguese course  
**Prepared by:** Khant Min Zaw (6845028)  
**Role:** Project Lead and Data Curator  
**Source:** https://archive.ics.uci.edu/dataset/320/student%2Bperformance

This notebook checks the dataset structure, missing values, duplicates,
data ranges, and potential outliers before cleaning.

In [1]:
from pathlib import Path

import pandas as pd

current_directory = Path.cwd()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

data_path = project_root / "data" / "raw" / "student_performance.csv"

df = pd.read_csv(data_path)

print(f"Dataset location: {data_path}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset location: C:\Users\KMZ\Projects\CSX2003\Statistics-superstars\data\raw\student_performance.csv
Rows: 649
Columns: 33


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      649 non-null    str  
 1   sex         649 non-null    str  
 2   age         649 non-null    int64
 3   address     649 non-null    str  
 4   famsize     649 non-null    str  
 5   Pstatus     649 non-null    str  
 6   Medu        649 non-null    int64
 7   Fedu        649 non-null    int64
 8   Mjob        649 non-null    str  
 9   Fjob        649 non-null    str  
 10  reason      649 non-null    str  
 11  guardian    649 non-null    str  
 12  traveltime  649 non-null    int64
 13  studytime   649 non-null    int64
 14  failures    649 non-null    int64
 15  schoolsup   649 non-null    str  
 16  famsup      649 non-null    str  
 17  paid        649 non-null    str  
 18  activities  649 non-null    str  
 19  nursery     649 non-null    str  
 20  higher      649 non-null    str  
 21  inte

In [7]:
missing_values = df.isna().sum()
duplicate_rows = df.duplicated().sum()

print("Total missing values:", missing_values.sum())
print("Completely duplicated rows:", duplicate_rows)

missing_values[missing_values > 0]

Total missing values: 0
Completely duplicated rows: 0


Series([], dtype: int64)

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,649.0,16.744222,1.218138,15.0,16.0,17.0,18.0,22.0
Medu,649.0,2.514638,1.134552,0.0,2.0,2.0,4.0,4.0
Fedu,649.0,2.306626,1.099931,0.0,1.0,2.0,3.0,4.0
traveltime,649.0,1.568567,0.748660,1.0,1.0,1.0,2.0,4.0
studytime,649.0,1.930663,0.829510,1.0,1.0,2.0,2.0,4.0
failures,649.0,0.221880,0.593235,0.0,0.0,0.0,0.0,3.0
famrel,649.0,3.930663,0.955717,1.0,4.0,4.0,5.0,5.0
freetime,649.0,3.180277,1.051093,1.0,3.0,3.0,4.0,5.0
goout,649.0,3.184900,1.175766,1.0,2.0,3.0,4.0,5.0
Dalc,649.0,1.502311,0.924834,1.0,1.0,1.0,2.0,5.0


In [9]:
important_columns = ["age", "absences", "G1", "G2", "G3"]

range_summary = pd.DataFrame({
    "minimum": df[important_columns].min(),
    "maximum": df[important_columns].max(),
    "mean": df[important_columns].mean().round(3),
    "median": df[important_columns].median(),
})

range_summary

,minimum,maximum,mean,median
age,15,22,16.744,17.0
absences,0,32,3.659,2.0
G1,0,19,11.399,11.0
G2,0,19,11.570,11.0
G3,0,19,11.906,12.0


In [10]:
validation_checks = pd.Series({
    "Correct row count": len(df) == 649,
    "Correct column count": len(df.columns) == 33,
    "No missing values": df.isna().sum().sum() == 0,
    "No duplicate rows": df.duplicated().sum() == 0,
    "Valid ages": df["age"].between(15, 22).all(),
    "Non-negative absences": df["absences"].ge(0).all(),
    "Valid G1 grades": df["G1"].between(0, 20).all(),
    "Valid G2 grades": df["G2"].between(0, 20).all(),
    "Valid G3 grades": df["G3"].between(0, 20).all(),
})

validation_checks

Correct row count        True
Correct column count     True
No missing values        True
No duplicate rows        True
Valid ages               True
Non-negative absences    True
Valid G1 grades          True
Valid G2 grades          True
Valid G3 grades          True
dtype: bool

In [11]:
q1 = df["absences"].quantile(0.25)
q3 = df["absences"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

absence_outliers = df[df["absences"] > upper_limit]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("IQR upper limit:", upper_limit)
print("Potential absence outliers:", len(absence_outliers))
print(
    "Flagged absence values:",
    sorted(absence_outliers["absences"].unique())
)

Q1: 0.0
Q3: 6.0
IQR: 6.0
IQR upper limit: 15.0
Potential absence outliers: 21
Flagged absence values: [np.int64(16), np.int64(18), np.int64(21), np.int64(22), np.int64(24), np.int64(26), np.int64(30), np.int64(32)]


In [12]:
print("Students with G3 equal to zero:", (df["G3"] == 0).sum())
print("\nSchool counts:")
print(df["school"].value_counts())

Students with G3 equal to zero: 15

School counts:
school
GP    423
MS    226
Name: count, dtype: int64


In [13]:
report_directory = project_root / "reports"
report_directory.mkdir(parents=True, exist_ok=True)

report = f"""DATA QUALITY REPORT
===================

Dataset: UCI Student Performance - Portuguese course
Prepared by: Khant Min Zaw (6845028)
Role: Project Lead and Data Curator

Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns
Total missing values: {df.isna().sum().sum()}
Completely duplicated rows: {df.duplicated().sum()}

Age range: {df["age"].min()} to {df["age"].max()}
Absence range: {df["absences"].min()} to {df["absences"].max()}
G1 range: {df["G1"].min()} to {df["G1"].max()}
G2 range: {df["G2"].min()} to {df["G2"].max()}
G3 range: {df["G3"].min()} to {df["G3"].max()}

IQR-flagged absence records: {len(absence_outliers)}
Students with G3 equal to zero: {(df["G3"] == 0).sum()}

Decision:
No records were removed during initial inspection.
There are no missing values or completely duplicated rows.
All inspected values are within the official documented ranges.
Absence values above the IQR threshold were retained because they are
plausible observations rather than confirmed data-entry errors.
G3 values equal to zero were retained because zero is a valid grade.
"""

report_path = report_directory / "data_quality_report.txt"
report_path.write_text(report, encoding="utf-8")

print(report)
print(f"Report saved to: {report_path}")

DATA QUALITY REPORT

Dataset: UCI Student Performance - Portuguese course
Prepared by: Khant Min Zaw (6845028)
Role: Project Lead and Data Curator

Dataset shape: 649 rows and 33 columns
Total missing values: 0
Completely duplicated rows: 0

Age range: 15 to 22
Absence range: 0 to 32
G1 range: 0 to 19
G2 range: 0 to 19
G3 range: 0 to 19

IQR-flagged absence records: 21
Students with G3 equal to zero: 15

Decision:
No records were removed during initial inspection.
There are no missing values or completely duplicated rows.
All inspected values are within the official documented ranges.
Absence values above the IQR threshold were retained because they are
plausible observations rather than confirmed data-entry errors.
G3 values equal to zero were retained because zero is a valid grade.

Report saved to: C:\Users\KMZ\Projects\CSX2003\Statistics-superstars\reports\data_quality_report.txt
